# Control: is a *well-posed* baseline also uninformative?

**The gap this closes.** Table I shows the MESOR term contributes +0.007 while the
residual contributes +0.095, and the paper reads that as evidence the mechanism is
within-session baseline normalisation rather than circadian correction.

A reviewer can object: the fitted MESOR reaches 25,811 ms against a true mean RR of
~840 ms. That parameter is numerically broken, not merely uninformative. Showing a
garbage number contributes nothing is trivially true — it does not establish that a
*sensible* per-subject baseline level is uninformative. As written, the paper tests
the negative claim (not circadian) but never the positive one (it is baseline
normalisation).

**This notebook supplies the missing control.** It replaces the two Cosinor
parameters (MESOR, amplitude) with a well-posed, non-degenerate per-subject
baseline — the session mean RR and session SD RR — and reruns the same
decomposition. Nothing else changes: same windows, same HRV features, same residual
features, same circadian encodings, same LOSO protocol.

```
Table I  (existing) :  HRV + circ  |  + MESOR, amplitude  |  + residual  |  + both
This run (new)      :  HRV + circ  |  + mean_RR, sd_RR    |  + residual  |  + both
```

**How to read the outcome.**

- If the session mean also contributes ~0.01, the objection dissolves: any
  session-level *level* term is uninformative once within-window deviation is
  present, degenerate or not. The paper's positive claim is then tested, not asserted.
- If the session mean contributes substantially more than the MESOR did, that is a
  real finding and the paper's framing needs to change before submission rather
  than after review.

Either result is worth having. Run it before deciding what to write.

---

**Prerequisites.** Run this in the same session as `notebook-rigorous-analysis.ipynb`
after sections A1–A5 and A3 (preprocessing). It reuses `wesad`, `wesad_cos`,
`hrv_features`, `resid_features`, `circ_features` and `xgb_loso_predictions`. You do
not need to rerun U0–U5. Approximate runtime: 8 LOSO runs at ~0.8 min each, plus
feature construction — under 15 minutes.

## 1. Dependency check

In [1]:
!pip install neurokit2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 4.3 MB/s eta 0:00:00


In [2]:
import os, json, pickle, warnings, itertools
import numpy as np, pandas as pd
from scipy.signal import welch
from scipy.optimize import curve_fit
from scipy import stats
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (f1_score, cohen_kappa_score, classification_report,
                             confusion_matrix, mean_absolute_error)
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.losses import Loss
import neurokit2 as nk
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
np.random.seed(42)
print("TF", tf.__version__, "GPU", len(tf.config.list_physical_devices('GPU'))>0)

DATA_PATH='/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE_PATH='/kaggle/working/processed'; os.makedirs(SAVE_PATH, exist_ok=True)
SUBJECT_IDS=[2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES=['relaxed','mild','moderate','high']
N_BOOT=10000
RNG=np.random.RandomState(42)

FEATURE_NAMES=['mean_rr','sdnn','rmssd','pnn50','cv_rr','vlf','lf','hf',
               'lf_hf','lf_nu','sd1','sd2','sd_ratio']
RESIDUAL_NAMES=['res_mean','res_std','res_max','res_trend','res_energy']
CIRC_NAMES=['sin_24h','cos_24h','sin_90m','cos_90m','cortisol']
print("Config loaded.")

TF 2.20.0 GPU True
Config loaded.


In [3]:
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl",'rb') as f:
        data=pickle.load(f,encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()

def extract_rr_from_ecg(ecg, fs=700):
    ecg=nk.ecg_clean(ecg.flatten(), sampling_rate=fs)
    _,info=nk.ecg_peaks(ecg, sampling_rate=fs); rp=info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs), (rp[:-1]+rp[1:])/2.0/fs, rp

def clean_rr(rr,ts):
    rr=rr.copy().astype(float); rr[(rr<=300)|(rr>=2000)]=np.nan
    for i in range(1,len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]):
            if abs(rr[i]-rr[i-1])/rr[i-1]>0.20: rr[i]=np.nan
    m=np.isnan(rr)
    if m.any(): rr[m]=np.interp(np.where(m)[0],np.where(~m)[0],rr[~m])
    return rr, ts.copy()

def align_temp(wt, rp, fe=700, ft=4.0):
    tap=np.interp(rp/fe, np.arange(len(wt))/ft, wt); return (tap[:-1]+tap[1:])/2.0

def labels_to_rr(labels, rp):
    out=[]
    for i in range(len(rp)-1):
        seg=labels[rp[i]:rp[i+1]]; v=seg[seg>0]
        out.append(0 if len(v)==0 else np.bincount(v).argmax())
    return np.array(out)

def hrv_features(w, fs=4.0):
    rr,diff=np.array(w),np.diff(w)
    mean_rr=np.mean(rr); sdnn=np.std(rr); rmssd=np.sqrt(np.mean(diff**2))
    pnn50=np.sum(np.abs(diff)>50)/len(diff)*100; cv=sdnn/mean_rr
    t=np.cumsum(rr)/1000.0; u=np.interp(np.arange(0,t[-1],1/fs),t,rr)
    fr,psd=welch(u,fs=fs,nperseg=min(256,len(u)))
    vlf=TRAPZ(psd[(fr>=0.003)&(fr<0.04)]); lf=TRAPZ(psd[(fr>=0.04)&(fr<0.15)]); hf=TRAPZ(psd[(fr>=0.15)&(fr<0.40)])
    lf_hf=lf/(hf+1e-8); lf_nu=lf/(lf+hf+1e-8)
    sd1=np.sqrt(0.5)*np.std(diff); sd2=np.sqrt(max(2*sdnn**2-0.5*np.var(diff),0)); sdr=sd1/(sd2+1e-8)
    return np.array([mean_rr,sdnn,rmssd,pnn50,cv,vlf,lf,hf,lf_hf,lf_nu,sd1,sd2,sdr])

def resid_features(rw):
    r=np.array(rw)
    return np.array([np.mean(r),np.std(r),np.max(np.abs(r)),np.polyfit(np.arange(len(r)),r,1)[0],np.sum(r**2)/len(r)])

def circ_features(ts):
    t,hour=ts%86400,(ts%86400)/3600.0
    cort=0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2)
    return np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),cort])

def cos_model(th,m,a,p): return m+a*np.cos((2*np.pi/24.0)*th+p)
def fit_cos(sig,ts,p0):
    th=(ts%86400)/3600.0
    try:
        popt,_=curve_fit(cos_model,th,sig,p0=p0,maxfev=10000)
        base=cos_model(th,*popt)
        r2=1-np.sum((sig-base)**2)/np.sum((sig-np.mean(sig))**2)
        return base,sig-base,popt[0],popt[1],r2
    except RuntimeError:
        return np.full_like(sig,np.mean(sig)),sig-np.mean(sig),float(np.mean(sig)),0.0,0.0

def roll_rmssd(rr):
    o=np.zeros(len(rr))
    for i in range(len(rr)):
        w=rr[max(0,i-10):i+10]; d=np.diff(w); o[i]=np.sqrt(np.mean(d**2)) if len(d)>1 else 0
    return o
def roll_sdnn(rr):
    o=np.zeros(len(rr))
    for i in range(len(rr)):
        w=rr[max(0,i-10):i+10]; o[i]=np.std(w) if len(w)>1 else 0
    return o
print("Helpers defined.")

Helpers defined.


In [4]:
wesad={}
for sid in SUBJECT_IDS:
    try:
        chest,wt,labels=load_subject(sid); ecg=chest['ECG'].flatten()
        rr,ts,rp=extract_rr_from_ecg(ecg); temp=align_temp(wt,rp)
        rr,ts=clean_rr(rr,ts); rl=labels_to_rr(labels,rp)
        keep=rl>0; rrk,tk,tsk,lk=rr[keep],temp[keep],ts[keep],rl[keep]
        new=np.zeros(len(lk),dtype=int); si=np.where(lk==2)[0]
        if len(si)>0:
            srr=rrk[si]; loc=[]
            for i in range(len(srr)):
                w=srr[max(0,i-15):i+15]; dd=np.diff(w); loc.append(np.sqrt(np.mean(dd**2)) if len(dd)>0 else 50)
            loc=np.array(loc); p33,p66=np.percentile(loc,33),np.percentile(loc,66)
            for i,idx in enumerate(si): new[idx]=(1 if loc[i]>=p66 else 2 if loc[i]>=p33 else 3)
        wesad[f'S{sid}']={'rr_ms':rrk,'temp':tk,'timestamps':tsk,'labels':new}
    except Exception as e: print(f"S{sid} FAIL {e}")
print(f"{len(wesad)} subjects")

wesad_cos={}
for sid,d in wesad.items():
    _,rr_res,mesor,amp,r2h=fit_cos(d['rr_ms'],d['timestamps'],[np.mean(d['rr_ms']),50.0,-1.5])
    _,temp_res,tm,ta,r2t=fit_cos(d['temp'],d['timestamps'],[np.mean(d['temp']),1.0,-1.5])
    wesad_cos[sid]={'rr_res':rr_res,'temp_res':temp_res,'mesor':mesor,'amplitude':amp,
                    'r2_hrv':r2h,'r2_temp':r2t,'temp_mesor':tm,'temp_amp':ta}
print("Cosinor fitted.")

15 subjects
Cosinor fitted.


In [5]:
def build_all(data, cos, window=120, step=5):
    Xseq,Xcirc,Xxgb,y,g,hourv,phasev=[],[],[],[],[],[],[]
    # decomposition columns kept separately for U1
    mesor_col, resid_cols, time_cols = [], [], []
    for sid,d in data.items():
        rr,temp=d['rr_ms'],d['temp']; labels,ts=d['labels'],d['timestamps']
        rr_res,temp_res=cos[sid]['rr_res'],cos[sid]['temp_res']
        mesor,amp=cos[sid]['mesor'],cos[sid]['amplitude']
        rn=(rr-np.mean(rr))/(np.std(rr)+1e-8)
        tn=(temp-np.mean(temp))/(np.std(temp)+1e-8)
        rrn=(rr_res-np.mean(rr_res))/(np.std(rr_res)+1e-8)
        trn=(temp_res-np.mean(temp_res))/(np.std(temp_res)+1e-8)
        rm,sd=roll_rmssd(rn),roll_sdnn(rn); hr=60000/(rr+1e-8)
        t0,t1=ts.min(),ts.max(); dur=max(t1-t0,1e-6)
        for s in range(0,len(rn)-window,step):
            e=s+window; lab=labels[s+window//2]; bi=min(s+window//2,len(ts)-1)
            seq=np.stack([rn[s:e],rm[s:e],sd[s:e],hr[s:e],rrn[s:e],tn[s:e],trn[s:e]],axis=-1)
            t=ts[bi]%86400; hour=t/3600.0
            circ=np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),
                0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2),
                np.sin(2*np.pi*(hour-23)/24),np.cos(2*np.pi*(hour-23)/24)])
            try:
                hf=hrv_features(rr[s:e]); rf=resid_features(rr_res[s:e]); cf=circ_features(ts[bi])
                xgbf=np.concatenate([hf,rf,np.array([mesor,amp]),cf])
            except Exception: continue
            Xseq.append(seq);Xcirc.append(circ);Xxgb.append(xgbf)
            y.append(lab);g.append(int(sid[1:]))
            hourv.append(ts[bi]-t0)                 # seconds since session start
            phasev.append((ts[bi]-t0)/dur)          # normalised phase 0..1
            mesor_col.append([mesor,amp])
            resid_cols.append(rf)
            time_cols.append(cf)
    return (np.array(Xseq,dtype=np.float32),np.array(Xcirc,dtype=np.float32),
            np.array(Xxgb),np.array(y,dtype=np.int32),np.array(g,dtype=np.int32),
            np.array(hourv),np.array(phasev),np.array(mesor_col),
            np.array(resid_cols),np.array(time_cols))

(X_seq,X_circ,X_xgb,y_all,groups,sess_sec,phase,
 MESOR_COL,RESID_COL,TIME_COL)=build_all(wesad,wesad_cos)
print("seq",X_seq.shape,"xgb",X_xgb.shape,"classes",np.bincount(y_all))
print("phase range",phase.min(),phase.max())

# HRV-only feature block = first 13 columns + circadian last 5 (no residual/mesor)
HRV_IDX=list(range(0,13))+list(range(20,25))    # 13 hrv + 5 circadian
print("HRV-only feature dim:",len(HRV_IDX),"| full dim:",X_xgb.shape[1])

seq (11846, 120, 7) xgb (11846, 25) classes [8594 1101 1080 1071]
phase range 0.0077978789368999 0.9915185703251734
HRV-only feature dim: 18 | full dim: 25


In [6]:
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma=gamma
    def call(self,yt,yp):
        yt=tf.cast(yt,tf.int32)
        ce=tf.keras.losses.sparse_categorical_crossentropy(yt,yp)
        pt=tf.reduce_sum(tf.one_hot(yt,4)*yp,axis=-1)
        return tf.pow(1.0-pt,self.gamma)*ce

def build_cnn(window=120,nch=7,ncirc=7,ncls=4):
    si=tf.keras.Input(shape=(window,nch),name='sequence')
    ci=tf.keras.Input(shape=(ncirc,),name='circadian')
    x=layers.Conv1D(64,7,padding='same',activation='relu')(si)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x)
    x=layers.Dropout(0.4)(x); a=layers.Attention()([x,x]); x=layers.GlobalAveragePooling1D()(a)
    c=layers.Dense(32,activation='relu')(ci); c=layers.Dense(16,activation='relu')(c)
    x=layers.Concatenate()([x,c]); x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    out=layers.Dense(ncls,activation='softmax')(x)
    return Model([si,ci],out,name='CNN_BiLSTM_Attn_v2')

def make_xgb():
    return XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.8,objective='multi:softprob',num_class=4,
        eval_metric='mlogloss',random_state=42,n_jobs=-1)

def xgb_loso_predictions(X, y, groups, cols=None):
    """Return pooled y_true, y_pred, y_proba, subject_ids in fold order."""
    logo=LeaveOneGroupOut()
    yt,yp,pp,subj=[],[],[],[]
    perfold={}
    for tr,te in logo.split(X,y,groups):
        s=int(np.unique(groups[te])[0])
        Xtr=X[tr][:,cols] if cols is not None else X[tr]
        Xte=X[te][:,cols] if cols is not None else X[te]
        sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
        m=make_xgb(); m.fit(Xtr,y[tr],sample_weight=compute_sample_weight('balanced',y[tr]),verbose=False)
        proba=m.predict_proba(Xte); pred=np.argmax(proba,axis=1)
        yt.extend(y[te]); yp.extend(pred); pp.extend(proba); subj.extend([s]*len(te))
        perfold[s]=f1_score(y[te],pred,average='macro',zero_division=0)
    return (np.array(yt),np.array(yp),np.array(pp),np.array(subj),perfold)
print("Models + LOSO helper defined.")

Models + LOSO helper defined.


In [7]:
# Fail loudly and immediately if the session is missing anything, rather than
# halfway through a 15-minute run.
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score

_needed = ['wesad', 'wesad_cos', 'hrv_features', 'resid_features',
           'circ_features', 'xgb_loso_predictions']
_missing = [n for n in _needed if n not in dir()]
assert not _missing, (
    f"missing from session: {_missing}. Run notebook-rigorous-analysis.ipynb "
    "sections A1-A5 (and A3 preprocessing) first.")

assert len(wesad) == 15, f"expected 15 subjects, found {len(wesad)}"
_k = list(wesad_cos)[0]
print("wesad keys      :", sorted(wesad[list(wesad)[0]].keys()))
print("wesad_cos keys  :", sorted(wesad_cos[_k].keys()))
print("subjects        :", len(wesad))
print("dependency check passed")

wesad keys      : ['labels', 'rr_ms', 'temp', 'timestamps']
wesad_cos keys  : ['amplitude', 'mesor', 'r2_hrv', 'r2_temp', 'rr_res', 'temp_amp', 'temp_mesor', 'temp_res']
subjects        : 15
dependency check passed


In [8]:
def build_all(data, cos, window=120, step=5):
    Xseq,Xcirc,Xxgb,y,g,hourv,phasev=[],[],[],[],[],[],[]
    # decomposition columns kept separately for U1
    mesor_col, resid_cols, time_cols = [], [], []
    for sid,d in data.items():
        rr,temp=d['rr_ms'],d['temp']; labels,ts=d['labels'],d['timestamps']
        rr_res,temp_res=cos[sid]['rr_res'],cos[sid]['temp_res']
        mesor,amp=cos[sid]['mesor'],cos[sid]['amplitude']
        rn=(rr-np.mean(rr))/(np.std(rr)+1e-8)
        tn=(temp-np.mean(temp))/(np.std(temp)+1e-8)
        rrn=(rr_res-np.mean(rr_res))/(np.std(rr_res)+1e-8)
        trn=(temp_res-np.mean(temp_res))/(np.std(temp_res)+1e-8)
        rm,sd=roll_rmssd(rn),roll_sdnn(rn); hr=60000/(rr+1e-8)
        t0,t1=ts.min(),ts.max(); dur=max(t1-t0,1e-6)
        for s in range(0,len(rn)-window,step):
            e=s+window; lab=labels[s+window//2]; bi=min(s+window//2,len(ts)-1)
            seq=np.stack([rn[s:e],rm[s:e],sd[s:e],hr[s:e],rrn[s:e],tn[s:e],trn[s:e]],axis=-1)
            t=ts[bi]%86400; hour=t/3600.0
            circ=np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),
                0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2),
                np.sin(2*np.pi*(hour-23)/24),np.cos(2*np.pi*(hour-23)/24)])
            try:
                hf=hrv_features(rr[s:e]); rf=resid_features(rr_res[s:e]); cf=circ_features(ts[bi])
                xgbf=np.concatenate([hf,rf,np.array([mesor,amp]),cf])
            except Exception: continue
            Xseq.append(seq);Xcirc.append(circ);Xxgb.append(xgbf)
            y.append(lab);g.append(int(sid[1:]))
            hourv.append(ts[bi]-t0)                 # seconds since session start
            phasev.append((ts[bi]-t0)/dur)          # normalised phase 0..1
            mesor_col.append([mesor,amp])
            resid_cols.append(rf)
            time_cols.append(cf)
    return (np.array(Xseq,dtype=np.float32),np.array(Xcirc,dtype=np.float32),
            np.array(Xxgb),np.array(y,dtype=np.int32),np.array(g,dtype=np.int32),
            np.array(hourv),np.array(phasev),np.array(mesor_col),
            np.array(resid_cols),np.array(time_cols))

(X_seq,X_circ,X_xgb,y_all,groups,sess_sec,phase,
 MESOR_COL,RESID_COL,TIME_COL)=build_all(wesad,wesad_cos)
print("seq",X_seq.shape,"xgb",X_xgb.shape,"classes",np.bincount(y_all))
print("phase range",phase.min(),phase.max())

# HRV-only feature block = first 13 columns + circadian last 5 (no residual/mesor)
HRV_IDX=list(range(0,13))+list(range(20,25))    # 13 hrv + 5 circadian
print("HRV-only feature dim:",len(HRV_IDX),"| full dim:",X_xgb.shape[1])

seq (11846, 120, 7) xgb (11846, 25) classes [8594 1101 1080 1071]
phase range 0.0077978789368999 0.9915185703251734
HRV-only feature dim: 18 | full dim: 25


In [9]:
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma=gamma
    def call(self,yt,yp):
        yt=tf.cast(yt,tf.int32)
        ce=tf.keras.losses.sparse_categorical_crossentropy(yt,yp)
        pt=tf.reduce_sum(tf.one_hot(yt,4)*yp,axis=-1)
        return tf.pow(1.0-pt,self.gamma)*ce

def build_cnn(window=120,nch=7,ncirc=7,ncls=4):
    si=tf.keras.Input(shape=(window,nch),name='sequence')
    ci=tf.keras.Input(shape=(ncirc,),name='circadian')
    x=layers.Conv1D(64,7,padding='same',activation='relu')(si)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x)
    x=layers.Dropout(0.4)(x); a=layers.Attention()([x,x]); x=layers.GlobalAveragePooling1D()(a)
    c=layers.Dense(32,activation='relu')(ci); c=layers.Dense(16,activation='relu')(c)
    x=layers.Concatenate()([x,c]); x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    out=layers.Dense(ncls,activation='softmax')(x)
    return Model([si,ci],out,name='CNN_BiLSTM_Attn_v2')

def make_xgb():
    return XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.8,objective='multi:softprob',num_class=4,
        eval_metric='mlogloss',random_state=42,n_jobs=-1)

def xgb_loso_predictions(X, y, groups, cols=None):
    """Return pooled y_true, y_pred, y_proba, subject_ids in fold order."""
    logo=LeaveOneGroupOut()
    yt,yp,pp,subj=[],[],[],[]
    perfold={}
    for tr,te in logo.split(X,y,groups):
        s=int(np.unique(groups[te])[0])
        Xtr=X[tr][:,cols] if cols is not None else X[tr]
        Xte=X[te][:,cols] if cols is not None else X[te]
        sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
        m=make_xgb(); m.fit(Xtr,y[tr],sample_weight=compute_sample_weight('balanced',y[tr]),verbose=False)
        proba=m.predict_proba(Xte); pred=np.argmax(proba,axis=1)
        yt.extend(y[te]); yp.extend(pred); pp.extend(proba); subj.extend([s]*len(te))
        perfold[s]=f1_score(y[te],pred,average='macro',zero_division=0)
    return (np.array(yt),np.array(yp),np.array(pp),np.array(subj),perfold)
print("Models + LOSO helper defined.")

Models + LOSO helper defined.


## 2. Build both feature matrices

Two 25-dimensional matrices, identical except for columns 18–19:

| cols | contents |
| --- | --- |
| 0–12 | 13 HRV time/frequency features |
| 13–17 | 5 residual-derived features |
| **18–19** | **MESOR + amplitude** *(variant A)* or **session mean RR + session SD RR** *(variant B)* |
| 20–24 | 5 circadian/time-of-day encodings |

Building both in the same loop guarantees the windows, labels and every other
feature are byte-identical between variants, so any difference in the results is
attributable to columns 18–19 and nothing else.

In [10]:
WINDOW, STEP = 120, 5      # matches the research pipeline behind Table I

def build_both(data, cos, window=WINDOW, step=STEP):
    Xa, Xb, y, g = [], [], [], []
    for sid, d in data.items():
        rr, ts, lab = d['rr_ms'], d['timestamps'], d['labels']
        rr_res = cos[sid]['rr_res']
        mesor, amp = cos[sid]['mesor'], cos[sid]['amplitude']

        # well-posed per-subject baseline: no curve fitting, cannot be degenerate
        sess_mean = float(np.mean(rr))
        sess_sd   = float(np.std(rr))

        for s in range(0, len(rr)-window, step):
            e = s + window
            mid = s + window//2
            bi = min(mid, len(ts)-1)
            try:
                hrv  = hrv_features(rr[s:e])
                res  = resid_features(rr_res[s:e])
                circ = circ_features(ts[bi])
            except Exception:
                continue
            Xa.append(np.concatenate([hrv, res, np.array([mesor, amp]),         circ]))
            Xb.append(np.concatenate([hrv, res, np.array([sess_mean, sess_sd]), circ]))
            y.append(lab[mid]); g.append(int(sid[1:]))
    return (np.array(Xa), np.array(Xb), np.array(y, dtype=int), np.array(g, dtype=int))

X_mesor, X_smean, y_all4, g_all4 = build_both(wesad, wesad_cos)

print("X_mesor", X_mesor.shape, " X_smean", X_smean.shape)
print("classes", np.bincount(y_all4, minlength=4))
print("\nsanity: columns 0-17 and 20-24 identical between variants:",
      np.allclose(np.delete(X_mesor, [18,19], axis=1),
                  np.delete(X_smean, [18,19], axis=1)))
print("sanity: columns 18-19 differ:",
      not np.allclose(X_mesor[:, 18:20], X_smean[:, 18:20]))

print("\nper-subject baseline terms (why the control is needed):")
print(f"{'subject':>8}{'MESOR (ms)':>14}{'session mean (ms)':>20}")
for sid in list(wesad)[:15]:
    print(f"{sid:>8}{wesad_cos[sid]['mesor']:>14.0f}{np.mean(wesad[sid]['rr_ms']):>20.1f}")

X_mesor (11846, 25)  X_smean (11846, 25)
classes [8594 1101 1080 1071]

sanity: columns 0-17 and 20-24 identical between variants: True
sanity: columns 18-19 differ: True

per-subject baseline terms (why the control is needed):
 subject    MESOR (ms)   session mean (ms)
      S2          4044               839.9
      S3         19382               982.9
      S4          6747               917.5
      S5          9857               866.1
      S6          4583               854.4
      S7          3110               859.2
      S8          5438               803.3
      S9          3533               773.6
     S10          1325               653.0
     S11         19425               676.8
     S13          6753               675.7
     S14         19916               682.1
     S15          4600               759.4
     S16         25811               709.9
     S17         15424               767.5


## 3. Run the decomposition on both variants

Variant A must reproduce Table I. If it does not, the pipeline has drifted and the
new row is not comparable to the published table — stop and diagnose before reading
anything into variant B.

In [11]:
C_HRV   = list(range(0, 13))
C_RESID = list(range(13, 18))
C_LEVEL = [18, 19]                 # MESOR+amp, or session mean+SD
C_CIRC  = list(range(20, 25))

CONFIGS = {
    'HRV + time (baseline)'   : C_HRV + C_CIRC,
    '+ level term'            : C_HRV + C_CIRC + C_LEVEL,
    '+ residual'              : C_HRV + C_CIRC + C_RESID,
    '+ level + residual'      : C_HRV + C_CIRC + C_LEVEL + C_RESID,
}

def decompose(X, y, g, tag):
    out = {}
    print("="*64); print(tag); print("="*64)
    for name, cols in CONFIGS.items():
        yt, yp, _, _, _ = xgb_loso_predictions(X, y, g, cols=cols)
        f1 = f1_score(yt, yp, average='macro', zero_division=0)
        k  = cohen_kappa_score(yt, yp, weights='quadratic')
        out[name] = (f1, k)
        print(f"{name:<26} F1={f1:.3f}  kappa={k:.3f}")
    base = out['HRV + time (baseline)'][0]
    out['_d_level'] = out['+ level term'][0] - base
    out['_d_resid'] = out['+ residual'][0]   - base
    print("-"*64)
    print(f"attributable to LEVEL term : {out['_d_level']:+.3f}")
    print(f"attributable to RESIDUAL   : {out['_d_resid']:+.3f}")
    print()
    return out

res_mesor = decompose(X_mesor, y_all4, g_all4, "VARIANT A - Cosinor MESOR + amplitude (reproduces Table I)")

# reproduction gate
_exp = {'HRV + time (baseline)': 0.551, '+ level term': 0.558,
        '+ residual': 0.646, '+ level + residual': 0.642}
print("reproduction check against published Table I:")
_ok = True
for k_, v_ in _exp.items():
    got = res_mesor[k_][0]
    hit = abs(got - v_) <= 0.005
    _ok &= hit
    print(f"  {k_:<26} expected {v_:.3f}  got {got:.3f}  {'OK' if hit else 'MISMATCH'}")
print("\nreproduction PASSED" if _ok else
      "\nreproduction FAILED - do not interpret variant B until this is resolved")

VARIANT A - Cosinor MESOR + amplitude (reproduces Table I)
HRV + time (baseline)      F1=0.551  kappa=0.668
+ level term               F1=0.558  kappa=0.696
+ residual                 F1=0.646  kappa=0.834
+ level + residual         F1=0.642  kappa=0.834
----------------------------------------------------------------
attributable to LEVEL term : +0.007
attributable to RESIDUAL   : +0.096

reproduction check against published Table I:
  HRV + time (baseline)      expected 0.551  got 0.551  OK
  + level term               expected 0.558  got 0.558  OK
  + residual                 expected 0.646  got 0.646  OK
  + level + residual         expected 0.642  got 0.642  OK

reproduction PASSED


In [12]:
res_smean = decompose(X_smean, y_all4, g_all4,
                      "VARIANT B - session mean RR + session SD (well-posed baseline)")

VARIANT B - session mean RR + session SD (well-posed baseline)
HRV + time (baseline)      F1=0.551  kappa=0.668
+ level term               F1=0.602  kappa=0.777
+ residual                 F1=0.646  kappa=0.834
+ level + residual         F1=0.643  kappa=0.838
----------------------------------------------------------------
attributable to LEVEL term : +0.051
attributable to RESIDUAL   : +0.096



## 4. Side-by-side and interpretation

In [13]:
print("="*78)
print("CONTROL RESULT: does a well-posed baseline behave like the degenerate one?")
print("="*78)
print(f"{'configuration':<26}{'A: MESOR':>16}{'B: session mean':>20}")
print("-"*78)
for name in CONFIGS:
    a, b = res_mesor[name][0], res_smean[name][0]
    print(f"{name:<26}{a:>16.3f}{b:>20.3f}")
print("-"*78)
dA_lvl, dA_res = res_mesor['_d_level'], res_mesor['_d_resid']
dB_lvl, dB_res = res_smean['_d_level'], res_smean['_d_resid']
print(f"{'level-term contribution':<26}{dA_lvl:>+16.3f}{dB_lvl:>+20.3f}")
print(f"{'residual contribution':<26}{dA_res:>+16.3f}{dB_res:>+20.3f}")
print(f"{'residual : level ratio':<26}{dA_res/max(abs(dA_lvl),1e-9):>15.1f}x{dB_res/max(abs(dB_lvl),1e-9):>19.1f}x")
print("="*78)

print()
if abs(dB_lvl) < 0.02:
    print("OUTCOME: the well-posed baseline is ALSO uninformative "
          f"({dB_lvl:+.3f}).")
    print("The reviewer objection dissolves - the result is not an artefact of the")
    print("degenerate Cosinor fit. Any session-level *level* term adds little once")
    print("within-window deviation is present. The paper's positive claim is now")
    print("tested rather than asserted. Add variant B as a row in Table I.")
elif dB_lvl >= 0.02 and dB_lvl < dB_res:
    print(f"OUTCOME: the well-posed baseline contributes MORE ({dB_lvl:+.3f}) than the")
    print(f"degenerate MESOR did ({dA_lvl:+.3f}), but still less than the residual")
    print(f"({dB_res:+.3f}). The direction of the paper's claim survives, but the")
    print("magnitude was overstated by using a broken parameter. Report both rows")
    print("and revise the wording - do not report only variant A.")
else:
    print(f"OUTCOME: the well-posed baseline contributes {dB_lvl:+.3f}, comparable to or")
    print(f"exceeding the residual ({dB_res:+.3f}). This CONTRADICTS the paper's")
    print("central claim as currently written. Do not submit without addressing it.")
    print("The +0.007 in Table I would then be an artefact of the degenerate fit,")
    print("not evidence about baseline terms in general.")

CONTROL RESULT: does a well-posed baseline behave like the degenerate one?
configuration                     A: MESOR     B: session mean
------------------------------------------------------------------------------
HRV + time (baseline)                0.551               0.551
+ level term                         0.558               0.602
+ residual                           0.646               0.646
+ level + residual                   0.642               0.643
------------------------------------------------------------------------------
level-term contribution             +0.007              +0.051
residual contribution               +0.096              +0.096
residual : level ratio               14.2x                1.9x

OUTCOME: the well-posed baseline contributes MORE (+0.051) than the
degenerate MESOR did (+0.007), but still less than the residual
(+0.096). The direction of the paper's claim survives, but the
magnitude was overstated by using a broken parameter. Report both r

## 5. Draft text for the paper

Only use the branch matching the outcome printed above. Fill the bracketed values
from the table; do not round in a direction that flatters the result.

**If the control is also uninformative (expected case):**

> To confirm that this reflects the uninformativeness of a session-level baseline
> rather than the degeneracy of the Cosinor fit, we repeated the decomposition with
> the MESOR and amplitude replaced by a well-posed per-subject baseline: the session
> mean and standard deviation of the RR interval. This term contributes [+0.0XX]
> macro-F1, against [+0.0XX] for the residual — the same imbalance observed with the
> fitted parameters. The result is therefore a property of session-level level terms
> in general, not an artefact of the degenerate fit.

Add as a row in Table I labelled `+ session mean/SD`, and add one clause to the
Limitations noting the control was run only on WESAD.

**If the control contributes materially more:** report both rows, and change the
Discussion from "the MESOR carries negligible information" to a statement scoped to
what was actually measured. Do not describe the degenerate-fit result as though it
generalised.